## 统计分析

通过指定统计分析字段，得到每个特征的p_value，所有的p_value计算都是基于Ttest计算。支持指定不同的分组`group`，例如train、val、test等分组统计。

对于两大类不同的特征

1. 离散特征，统计数量以及占比。
2. 连续特征，统计均值、方差。

In [1]:
import pandas as pd
import numpy as np
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.utils import print_join_info

task = get_param_in_cwd('task_column') or ['label']
p_value = get_param_in_cwd('p_value') or 0.05
# 修改成自己临床数据的文件。
test_data = pd.read_csv(get_param_in_cwd('clinic_file'), dtype={'ID': str}).drop_duplicates('ID')
stats_columns_settings = get_param_in_cwd('stats_columns')
continuous_columns_settings = get_param_in_cwd('continuous_columns')
mapping_columns_settings = get_param_in_cwd('mapping_columns')
test_data = test_data[[c for c in test_data.columns if c != task]]
survival = pd.read_csv(get_param_in_cwd('label_file')).drop_duplicates('ID')
# group_info = pd.merge(group_info, survival, on='ID', how='inner')
print_join_info(test_data, survival)
test_data = pd.merge(test_data, survival, left_on='ID', right_on='ID', how='inner')
test_data

[2025-09-12 22:06:33 - __init__.py:  63]	WARNING	存在ID特征不完全匹配的问题！在左边不在右边的ID：['J07A0946', 'J07A0980', 'J07A0987', 'J07A0999', 'J07A1023', 'J07A1076', 'J07A1098', 'J07A3073', 'J07A3081', 'J07A3093', 'J07A3112', 'J07A3159', 'J07A3171']；在右边不在左边的ID：[]


,ID,number of lymph nodes,positive number,T,N,M,AJCC,HER2,age,event,duration,group
0,J07A3054,14.0,0.0,1.0,0.0,0,1.0,阳,56,0,132,train
1,J07A3055,15.0,0.0,2.0,0.0,0,2.0,阳,51,0,131,train
2,J07A3056,25.0,9.0,2.0,3.0,0,3.0,阴,39,1,78,train
3,J07A3057,18.0,2.0,1.0,1.0,0,2.0,阴,60,0,131,train
4,J07A3058,17.0,0.0,1.0,0.0,0,1.0,阴,55,0,131,test
...,...,...,...,...,...,...,...,...,...,...,...,...
273,J07A0934,13.0,0.0,2.0,0.0,M0,2.0,阴,37,0,149,train
274,J07A0939,21.0,21.0,3.0,3.0,M0,3.0,阳,50,0,147,train
275,J07A1062,10.0,2.0,1.0,1.0,M0,2.0,阳,46,0,119,test
276,J07A1080,11.0,0.0,1.0,0.0,M0,1.0,NaN,65,0,115,test


In [2]:
test_data['group'].value_counts()

train    194
test      84
Name: group, dtype: int64

# 特征名称处理

去掉所有特征名称中的特殊字符。

In [3]:
import re

def map_cnames(x):
    x = re.split('[（|(]', x)[0]
    x = x.replace('-', '_').replace(' ', '_').replace('>', '').replace('/', '_')
    return x.strip()

test_data.columns = list(map(map_cnames, test_data.columns))
test_data.columns

Index(['ID', 'number_of_lymph_nodes', 'positive_number', 'T', 'N', 'M', 'AJCC',
       'HER2', 'age', 'event', 'duration', 'group'],
      dtype='object')

# 分析数据

获取待分析的特征列名，如未制定，自动侦测。

In [4]:
stats_columns = [c for c in stats_columns_settings or list(test_data.columns[1:-3]) if c not in ['']]
test_data = test_data.copy()[['ID'] + stats_columns + list(test_data.columns[-3:-1]) +['group']]
test_data#['group'].value_counts()

,ID,number_of_lymph_nodes,positive_number,T,N,M,AJCC,HER2,age,event,duration,group
0,J07A3054,14.0,0.0,1.0,0.0,0,1.0,阳,56,0,132,train
1,J07A3055,15.0,0.0,2.0,0.0,0,2.0,阳,51,0,131,train
2,J07A3056,25.0,9.0,2.0,3.0,0,3.0,阴,39,1,78,train
3,J07A3057,18.0,2.0,1.0,1.0,0,2.0,阴,60,0,131,train
4,J07A3058,17.0,0.0,1.0,0.0,0,1.0,阴,55,0,131,test
...,...,...,...,...,...,...,...,...,...,...,...,...
273,J07A0934,13.0,0.0,2.0,0.0,M0,2.0,阴,37,0,149,train
274,J07A0939,21.0,21.0,3.0,3.0,M0,3.0,阳,50,0,147,train
275,J07A1062,10.0,2.0,1.0,1.0,M0,2.0,阳,46,0,119,test
276,J07A1080,11.0,0.0,1.0,0.0,M0,1.0,NaN,65,0,115,test


# 特征队列映射

所有需要进行特征映射的队列，range未制定，可以进行自动判断。

In [5]:
mapping_columns = mapping_columns_settings or [c for c in test_data.columns[1:-3] if test_data[c].dtype == object]
mapping_columns

['M', 'HER2']

# 数据映射

针对所有非数值形式的数据，可以进行类别映射。

In [6]:
from onekey_algo.custom.utils import map2numerical

data, mapping = map2numerical(test_data, mapping_columns=mapping_columns)
mapping

{'M': {'0': 0, 'M0': 1}, 'HER2': {'阳': 0, '阴': 1}}

In [7]:
data.dtypes

ID                        object
number_of_lymph_nodes    float64
positive_number          float64
T                        float64
N                        float64
M                          int64
AJCC                     float64
HER2                     float64
age                        int64
event                      int64
duration                   int64
group                     object
dtype: object

# 连续特征列

自动识别所有可能的连续特征列。如果列不是整数，或者列的元素超过5个，则呗认定为连续特征。

In [8]:
from onekey_algo.custom.components.comp1 import fillna

test_data = fillna(test_data, fill_mod='50%')
continuous_columns = []
for col in test_data.columns:
    if test_data[col].apply(lambda x: x.is_integer() if isinstance(x, float) else False).all():
        test_data[col] = test_data[col].astype(int)

for c in stats_columns:
#     print(c, np.unique(test_data[c]), test_data[c].dtype)
    if len(np.unique(test_data[c])) > 5 or not np.int8 <= test_data[c].dtype <= np.int64:
        continuous_columns.append(c)
        
continuous_columns = continuous_columns_settings or continuous_columns
continuous_columns = [c for c in continuous_columns if c not in ('differentation')]
# continuous_columns = []

In [9]:
continuous_columns

['number_of_lymph_nodes', 'positive_number', 'age']

# 缺失值填充

In [10]:
import os
os.makedirs('data', exist_ok=True)
data = test_data
data.to_csv('data/clinical.csv', index=False)
data

,ID,number_of_lymph_nodes,positive_number,T,N,M,AJCC,HER2,age,event,duration,group
0,J07A3054,14,0,1,0,0,1,0,56,0,132,train
1,J07A3055,15,0,2,0,0,2,0,51,0,131,train
2,J07A3056,25,9,2,3,0,3,1,39,1,78,train
3,J07A3057,18,2,1,1,0,2,1,60,0,131,train
4,J07A3058,17,0,1,0,0,1,1,55,0,131,test
...,...,...,...,...,...,...,...,...,...,...,...,...
273,J07A0934,13,0,2,0,1,2,1,37,0,149,train
274,J07A0939,21,21,3,3,1,3,0,50,0,147,train
275,J07A1062,10,2,1,1,1,2,0,46,0,119,test
276,J07A1080,11,0,1,0,1,1,1,65,0,115,test


In [11]:
data['group'].value_counts()

train    194
test      84
Name: group, dtype: int64

### 统计分析

支持两种格式数据，分别对应`pretty`参数的`True`和`False`, 当为`True`时，输出的是表格模式，反之则为dict数据。

```python
def clinic_stats(data: DataFrame, stats_columns: Union[str, List[str]], label_column='label',
                 group_column: str = None, continuous_columns: Union[str, List[str]] = None,
                 pretty: bool = True) -> Union[dict, DataFrame]:
    """

    Args:
        data: 数据
        stats_columns: 需要统计的列名
        label_column: 二分类的标签列，默认`label`
        group_column: 分组统计依据，例如区分训练组、测试组、验证组。
        continuous_columns: 那些列是连续变量，连续变量统计均值方差。
        pretty: bool, 是否对结果进行格式美化。

    Returns:
        stats DataFrame or json

    """
```

In [12]:
from onekey_algo.custom.components.stats import clinic_stats

pd.set_option('display.max_rows', None)
stats_tr = clinic_stats(data, 
                     stats_columns= stats_columns,
                     label_column='group', 
                     group_column=None, 
                     continuous_columns= continuous_columns, 
                     pretty=True, verbose=False)
# stats_tv['test'] = stats_tr['-label=test']
stats_tr.to_csv('data/stats_all.csv', index=False, encoding='utf_8_sig')
stats_tr

,feature_name,-label=ALL,-label=test,-label=train,pvalue
0,number_of_lymph_nodes,12.97±4.89,13.42±5.60,12.77±4.55,0.584
1,positive_number,2.46±3.71,2.27±3.13,2.54±3.94,0.836
2,age,54.71±12.43,54.17±12.43,54.94±12.46,0.586
0,T,,,,0.449
1,1,88(31.65),31(36.90),57(29.38),
2,2,174(62.59),49(58.33),125(64.43),
3,3,16(5.76),4(4.76),12(6.19),
4,N,,,,0.986
5,0,127(45.68),38(45.24),89(45.88),
6,1,66(23.74),21(25.00),45(23.20),
